In [ ]:
# CLEANING UP THE L1B HDR DATA COMPILED CSV & ADDING EXTRA INFO LIKE BAD BDEs, IF THE FILES ARE IN USGS ETC 

import pandas as pd 
import re
import numpy as np 
import os
import glob
from astropy.io import fits

#df = pd.read_csv("20260310_darkcurrent/l1b_rdn_hdr_info.csv")

In [ ]:
df['obs_date'] = pd.to_datetime(df['id'].str.extract(r'(\d{8})T')[0], format='%Y%m%d').dt.date
df['obs_time'] = df['id'].str.extract(r'T(\d{6})')[0].apply(lambda x: f"{x[:2]}:{x[2:4]}:{x[4:]}")

df['dark_signal_id'] = df['cal_dark_signal_image'].str.extract(r'(M3[GT]\d{8}T\d{6})')
df['bad_detector_map_id'] = df['cal_bad_detector_map'].str.extract(r'(M3[GT]\d{8}T\d{6})')
df['flat_field_id'] = df['cal_flat_field_image'].str.extract(r'(M3[GT]\d{8}T\d{6})')


In [ ]:
df['dark_signal_version'] = df['cal_dark_signal_image'].str.extract(r'(V\d+)')
df['bad_detector_map_version'] = df['cal_bad_detector_map'].str.extract(r'(V\d+)')
df['flat_field_version'] = df['cal_flat_field_image'].str.extract(r'(V\d+)')

In [ ]:
df = df.rename(columns={'id': 'obs_id'})

In [ ]:
df = df.drop(columns=['cal_dark_signal_image', 'cal_bad_detector_map','cal_flat_field_image', 'obs_filename', 'Unnamed: 0'])

In [ ]:
temp_df = pd.read_fwf('/home/bekah/m3_papers/20260310_darkcurrent/m3_detector_temperature.tab')

df = df.merge(temp_df.rename(columns={'id': 'dark_signal_id', 'temp': 'dark_signal_temp'}), on='dark_signal_id', how='left')
df = df.merge(temp_df.rename(columns={'id': 'bad_detector_map_id', 'temp': 'bad_detector_map_temp'}), on='bad_detector_map_id', how='left')
df = df.merge(temp_df.rename(columns={'id': 'flat_field_id', 'temp': 'flat_field_temp'}), on='flat_field_id', how='left')

In [ ]:

df['obs_date'] = pd.to_datetime(df['obs_date'])

conditions = [
    (df['obs_date'] >= '2008-11-18') & (df['obs_date'] <= '2009-01-24'),
    (df['obs_date'] >= '2009-01-09') & (df['obs_date'] <= '2009-02-14'),
    (df['obs_date'] >= '2009-04-15') & (df['obs_date'] <= '2009-04-27'),
    (df['obs_date'] >= '2009-05-13') & (df['obs_date'] <= '2009-05-16'),
    (df['obs_date'] >= '2009-05-20') & (df['obs_date'] <= '2009-07-09'),
    (df['obs_date'] >= '2009-07-12') & (df['obs_date'] <= '2009-08-16'),
]

labels = ['OP1A', 'OP1B', 'OP2A', 'OP2B', 'OP2C1', 'OP2C2']

df['obs_period'] = np.select(conditions, labels, default='Unknown')

In [ ]:
df

In [ ]:
docs = pd.read_csv("collection_document_inventory.csv")

In [ ]:
docs['id'] = docs['urn:nasa:pds:ch1_m3:document:document_readme::1.0'].str.split(":").str[5]
docs['type'] = docs['id'].str.split("_").str[1]
docs['id'] = docs['id'].str.split("_").str[0]


In [ ]:
df['flat_in_usgs'] = df['flat_field_id'].isin(docs[docs['type']=='ff']['id'].str.upper())
df['bde_in_usgs'] = df['bad_detector_map_id'].isin(docs[docs['type']=='bde']['id'].str.upper())


In [ ]:
df[df['bde_in_usgs']==False]['obs_id'].unique()

In [ ]:
df[df['flat_in_usgs']==False]['obs_id'].unique()

In [ ]:
data = pd.read_csv("collection_data_inventory.csv")
data['id'] = data['urn:nasa:pds:ch1_m3:data:data_readme::1.0'].str.split(":").str[5]
data['type'] = data['id'].str.split("_").str[1]
data['id'] = data['id'].str.split("_").str[0]
data

In [ ]:
df['dark_in_usgs'] = df['dark_signal_id'].isin(data[data['type']=='l0']['id'].str.upper())
df['cal_l0_in_usgs'] = df['obs_id'].isin(data[data['type']=='l0']['id'].str.upper())

# these are both all False 

In [ ]:
len(df['bad_detector_map_id'].unique())

In [ ]:


bde_folder = '/home/bekah/m3_papers/20260311_bde_darkcurrent/bde_fits'

records = []

for filepath in glob.glob(os.path.join(bde_folder, '*.fits')):
    filename = os.path.basename(filepath)
    bde_id = filename.split('_bde.fits')[0].upper()  
    with fits.open(filepath) as hdul:
        data = hdul[0].data
        nonzero_count = int(np.count_nonzero(data))
    
    records.append({
        'bde_id': bde_id,
        'flagged_pixels_in_bde': nonzero_count
    })

bde_pixel_df = pd.DataFrame(records)

In [ ]:
df = df.merge(bde_pixel_df, left_on='bad_detector_map_id', right_on='bde_id', how='left')

In [ ]:
def bde_flag_coverage(row):
    if row['obs_type'] == 'T':
        return row['flagged_pixels_in_bde'] / (260 * 640)
    elif row['obs_type'] == 'G':
        return row['flagged_pixels_in_bde'] / (86 * 320)
    else:
        return np.nan

df['bde_flag_coverage'] = df.apply(bde_flag_coverage, axis=1)

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
plt.scatter(df['obs_temperature'], df['bde_flag_coverage'], s=.5, c=df['bad_bde'])
plt.xlabel("temp (K)") 
plt.ylabel("portion of image flagged") 

In [ ]:
df.to_csv("obs_cal_info.csv")

In [ ]:
df = pd.read_csv("obs_cal_info.csv")

In [ ]:
# from looking at BDEs by hand 

bad_bdes = ['M3G20090515T043023', 'M3G20090601T015551', 'M3G20090601T061122', 'M3G20090601T102652']

In [ ]:
bad_bdes = ['M3G20090515T043023', 'M3G20090601T015551', 'M3G20090601T061122', 'M3G20090601T102652']

df['bad_bde'] = df['bad_detector_map_id'].isin(bad_bdes)
df['bad_dark'] = df['dark_signal_id'].isin(bad_bdes)

In [ ]:
df[df['bad_dark']==True]['obs_id'].unique()

In [ ]:
len(df[df['bad_bde']==True]['obs_id'].unique())

In [ ]:
df[df['bad_dark']==True]

In [ ]:
df[df['bad_bde']==True]

In [ ]:
df[df['bde_flag_coverage']>.4]

In [ ]:
df[df['obs_id'] == 'M3G20090626T142653']

In [ ]:

from pathlib import Path
import numpy as np
from astropy.io import fits
from PIL import Image

fits_file = 'm3g20090626t142653_l2_rfl.fits'
out_fits = 'm3g20090626t142653_l2_rfl_flat.fits'
with fits.open(fits_file) as hdul:
        data = hdul[0].data

bands, rows, cols = data.shape

tmp = data.transpose(1, 0, 2)  
image = tmp

fits.writeto(out_fits, image, overwrite=True)